In [2]:
from math import sqrt

import pandas as pd
import numpy as np
from pathlib import Path
import pyodbc 
import sqlalchemy
from sqlalchemy.engine import url
from sqlalchemy.engine.url import URL
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
#fit LSTM Model
import sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import keras
import tensorflow as tf
from tensorflow.keras.layers import LSTM
from keras.models import Sequential
from keras.layers import Dense



In [3]:
#!python.exe -m pip install --upgrade pip


**Data Overview & Preprocessing**

understanding load and explore the dataset.
Clean the data by handling missing values, outliers, and other anomalies.
Identify relevant features that might impact myopia progression.

In [3]:
p = Path.home()
p

path = r'myopia_forecast.xlsx'
df = pd.read_excel(path)

In [27]:
df = pd.read_sql("SELECT * FROM forecasting_ametropia", engine)

In [28]:
df.head()

In [29]:
df.info()

In [30]:
df.shape

In [31]:
df.describe()

In [32]:
from summarytools import dfSummary
dfSummary(df,is_collapsible = True)

In [33]:
df_hyperopia = df[df['ametropia']=='hyperopia']
df_myopia = df[df['ametropia']=='myopia']
df_highmyopia = df[df['ametropia']=='high myopia']
# df_premyopia = df[df['ametropia']=='pre-myopia']

In [34]:
data_hyperopia = df_hyperopia.groupby('age')[['age', 'SER_right', 'SER_left']].mean()
fig = px.line(data_hyperopia, x='age', y=['SER_right', 'SER_left'], markers=True)
fig.show()

age<45 ==> SER_left higher than SER_right
45>age<80 ==> SER_left==SER_right
age>80 ==> SER_left != SER_right

In [36]:
data_myopia = df_myopia.groupby('age')[['age', 'SER_right', 'SER_left']].mean()
fig = px.line(data_myopia, x='age', y=['SER_right', 'SER_left'], markers=True)
fig.show()

59>age<90 ==> SER_left != SER_right
age>90 ==> SER_left != SER_right le gap se creuse

In [38]:
data_highmyopia = df_highmyopia.groupby('age')[['age', 'SER_right', 'SER_left']].mean()
fig = px.line(data_highmyopia, x='age', y=['SER_right', 'SER_left'], markers=True)
fig.show()

age<13 ==> SERT_left higer then SER_right
13>age<80 ==> superposition 
age>80 huge gap between left and right

In [40]:
fig = px.box(df, x="ametropia", y="SER_left")
fig.show()

In [41]:
fig = px.box(df, x="ametropia", y="SER_right")
fig.show()

In [42]:
data = df.groupby('ametropia')[['age', 'SER_right', 'SER_left']].mean()#.query("SER_right<0")
data.head()

In [46]:
df.columns

In [ ]:
# Conversion des colonnes "object" en valeurs numériques
label_encoder = LabelEncoder()
df['ametropia_encoded'] = label_encoder.fit_transform(df['ametropia'])
df['gender_encoded'] = label_encoder.fit_transform(df['Gender'])
df['Country_encoded'] = label_encoder.fit_transform(df['Country'])

In [51]:
df.head()

In [53]:
df.info()

In [54]:
df_cat= df.select_dtypes(include='object')
df_num = df.select_dtypes(include=['float', 'int'])

In [78]:
#data correlation
corr_matrix = df_num.corr(method='spearman', min_periods=1) #la méthose "Spearman" me paraît la plus pertinente 
                                                                    #pour obtenir des résultats optimaux. 
# sns.heatmap(corr_matrix, annot=False)
cmap = sns.diverging_palette(220, 10, as_cmap=True)
matrix = sns.heatmap(corr_matrix, cmap=cmap, cbar_kws={"shrink": .5}, linewidths=.5)
plt.show()

**Data Preprocessing : fill na and take care of outliers**

In [56]:
#visualise outliers
fig_scatter = px.scatter(df, x='age', y=['SER_right', 'SER_left'])
fig_scatter.show()

In [57]:
#identify outliers 

# Compute Z-scores for CreditScore
df_num["Z_score1"] = (df_num["SER_right"] - df_num["SER_right"].mean()) / df_num["SER_right"].std()

# Detect outliers (Z-score > 3)
outliers = df_num[df_num["Z_score1"].abs() > 3]

# Display outliers
print("Outliers:")
print(outliers[["annee", "SER_right", "Z_score1"]])

In [58]:
# Visualize the SER_right distribution
plt.hist(df_num["Z_score1"], bins=20, rwidth=0.8)
plt.xlabel("Z_score1")
plt.ylabel("Count")
plt.title("Histogram - Z_score1")
plt.show()

In [59]:
# Compute Z-scores for CreditScore
df_num["Z_score2"] = (df_num["SER_left"] - df_num["SER_left"].mean()) / df_num["SER_left"].std()

# Detect outliers (Z-score > 3)
outliers = df_num[df_num["Z_score2"].abs() > 3]

# Display outliers
print("Outliers:")
print(outliers[["annee", "SER_left", "Z_score2"]])

In [60]:
# Visualize the SER_right distribution
plt.hist(df_num["SER_left"], bins=20, rwidth=0.8)
plt.xlabel("SER_left")
plt.ylabel("Count")
plt.title("Histogram - SER_left")
plt.show()

In [63]:
display(
df.isna().sum()
)

In [ ]:
#capper

In [ ]:
#fillna for numeric column  

df['LeftAddition'] = df['LeftAddition'].fillna(0)
df['RightAddition'] = df['RightAddition'].fillna(0)
df['LeftCylinder'] = df['LeftCylinder'].fillna(0)
df['RightCylinder'] = df['RightCylinder'].fillna(0)
# df['astigmatism'] = df['astigmatism'].fillna("Emmetrope")
df['LeftAxis'] = df['LeftAxis'].fillna(0)
df['RightAxis'] = df['RightAxis'].fillna(0)

df_mean_age = df['age'].mean()
df['age'] = df['age'].fillna(df_mean_age)

df_mode_Country = df['Country'].mode()
df['Country'] = df['Country'].fillna(df_mode_Country)

#df_median_gender = df['Gender_Id'].median()
#df['Gender_Id'] = df['Gender_Id'].fillna(df_median_gender)

In [68]:
df.head()

In [69]:
df.isna().sum()

In [70]:
#change the data type of the date column to fit the model needs
df['annee'] = pd.to_datetime(df['annee'], format='%Y')

In [71]:
#pass the date to index
df = df.set_index('annee')

In [ ]:
#create clusters by type of myopia
df_hyperopia = df[df['ametropia']=='hyperopia'].reset_index(drop=True)
df_highmyopia = df[df['ametropia']=='high myopia'].reset_index(drop=True)
df_myopia = df[df['ametropia']=='low myopia'].reset_index(drop=True)
# df_premyopia = df[df['ametropia']=='pre-myopia'].reset_index(drop=True)

In [73]:
df.head()

In [74]:
df.columns

In [75]:
df1 = df[['age', 'SER_left', 'SER_right']]
df1.head()

In [76]:
df1.shape

In [77]:
values = df1.values
# specify columns to plot
groups = [1, 2]
i = 1
#plot each column
plt.figure()
for group in groups:
    plt.subplot(len(groups), 1, i)
    plt.plot(values[:, group])
    plt.title(df1.columns[group], y=0.5, loc='right')
    i += 1
plt.show()

Model : LSTM

data preparation : coming